# 01 - Data Ingestion & Data Profiling

## Road Safety Data Pipeline

**Project:** Traffic Accident Analysis  
**Analytics Approach:** Descriptive Analytics  
**Framework:** Apache Spark (PySpark)  
**Environment:** VS Code - Jupyter Notebook

### Objectives
1. Menyiapkan environment Apache Spark.
2. Membaca dataset kecelakaan lalu lintas.
3. Memeriksa struktur dan karakteristik dataset.
4. Mengidentifikasi permasalahan kualitas data sebelum cleaning.


## 2 — Import library

In [1]:
import os
import sys
import findspark
import pyspark

import pandas as pd

from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    when,
    isnan,
    trim,
    lower
)

print("Python version:", sys.version)
print("PySpark version:", pyspark.__version__)
print("Pandas version:", pd.__version__)

Python version: 3.12.4 (tags/v3.12.4:8e8a4ba, Jun  6 2024, 19:30:16) [MSC v.1940 64 bit (AMD64)]
PySpark version: 4.3.0
Pandas version: 2.2.3


## 3 — Inisialisasi Apache Spark

In [3]:
# Inisialisasi findspark
findspark.init()

# Membuat Spark Session
spark = (
    SparkSession.builder
    .appName("RoadSafetyDataPipeline")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark berhasil dijalankan!")
print("Spark Version:", spark.version)
print("Application Name:", spark.sparkContext.appName)

Spark berhasil dijalankan!
Spark Version: 4.3.0
Application Name: RoadSafetyDataPipeline


## 4 — Menentukan direktori proyek

In [4]:
# Direktori utama repository
PROJECT_ROOT = Path.cwd().parent

# Lokasi dataset mentah
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "traffic_accidents.csv"

print("Project Root :", PROJECT_ROOT)
print("Dataset Path :", RAW_DATA_PATH)
print("File Exists  :", RAW_DATA_PATH.exists())

Project Root : d:\UMM\SEMESTER 7\Analisis Big Data\UAS\road-safety-data-pipeline
Dataset Path : d:\UMM\SEMESTER 7\Analisis Big Data\UAS\road-safety-data-pipeline\data\raw\traffic_accidents.csv
File Exists  : True


## 5 — Data Ingestion

In [5]:
# Membaca dataset CSV
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "PERMISSIVE")
    .csv(str(RAW_DATA_PATH))
)

print("Dataset berhasil dibaca!")

Dataset berhasil dibaca!


## 6 — Melihat data awal

In [6]:
# Menampilkan 10 baris pertama
df_raw.show(10, truncate=False)

+----------------------+----------------------+-----------------+----------------------+----------------+-------------------------------+------------------+--------------------+-----------+--------------------------------+----------------------+-------------+-----------------------------------+---------+------------------------+--------------+--------------+-----------------------+---------------------------+-----------------------------+----------------------+----------+-----------------+-----------+
|crash_date            |traffic_control_device|weather_condition|lighting_condition    |first_crash_type|trafficway_type                |alignment         |roadway_surface_cond|road_defect|crash_type                      |intersection_related_i|damage       |prim_contributory_cause            |num_units|most_severe_injury      |injuries_total|injuries_fatal|injuries_incapacitating|injuries_non_incapacitating|injuries_reported_not_evident|injuries_no_indication|crash_hour|crash_day_of_week

## 7 — Memeriksa dimensi dataset

In [7]:
total_rows = df_raw.count()
total_columns = len(df_raw.columns)

print("Jumlah Baris   :", total_rows)
print("Jumlah Kolom   :", total_columns)
print("Total Data     :", total_rows * total_columns)

Jumlah Baris   : 209306
Jumlah Kolom   : 24
Total Data     : 5023344


## 8 — Memeriksa struktur dan tipe data

In [8]:
# Menampilkan skema dataset
df_raw.printSchema()

# Daftar nama kolom
print("Daftar Kolom:")
for index, column_name in enumerate(df_raw.columns, start=1):
    print(f"{index}. {column_name}")

root
 |-- crash_date: string (nullable = true)
 |-- traffic_control_device: string (nullable = true)
 |-- weather_condition: string (nullable = true)
 |-- lighting_condition: string (nullable = true)
 |-- first_crash_type: string (nullable = true)
 |-- trafficway_type: string (nullable = true)
 |-- alignment: string (nullable = true)
 |-- roadway_surface_cond: string (nullable = true)
 |-- road_defect: string (nullable = true)
 |-- crash_type: string (nullable = true)
 |-- intersection_related_i: string (nullable = true)
 |-- damage: string (nullable = true)
 |-- prim_contributory_cause: string (nullable = true)
 |-- num_units: integer (nullable = true)
 |-- most_severe_injury: string (nullable = true)
 |-- injuries_total: double (nullable = true)
 |-- injuries_fatal: double (nullable = true)
 |-- injuries_incapacitating: double (nullable = true)
 |-- injuries_non_incapacitating: double (nullable = true)
 |-- injuries_reported_not_evident: double (nullable = true)
 |-- injuries_no_indi

## 9 — Statistik deskriptif awal

In [9]:
df_raw.describe().show(truncate=False)

+-------+----------------------+----------------------+------------------------+------------------+----------------+---------------+---------------------+--------------------+-----------------+--------------------------------+----------------------+------------+-----------------------+------------------+---------------------+-------------------+--------------------+-----------------------+---------------------------+-----------------------------+----------------------+------------------+------------------+-----------------+
|summary|crash_date            |traffic_control_device|weather_condition       |lighting_condition|first_crash_type|trafficway_type|alignment            |roadway_surface_cond|road_defect      |crash_type                      |intersection_related_i|damage      |prim_contributory_cause|num_units         |most_severe_injury   |injuries_total     |injuries_fatal      |injuries_incapacitating|injuries_non_incapacitating|injuries_reported_not_evident|injuries_no_indicati

## 10 — Pemeriksaan NULL dan nilai kosong

In [10]:
null_summary = df_raw.select([
    count(
        when(
            col(c).isNull() |
            (trim(col(c).cast("string")) == ""),
            c
        )
    ).alias(c)
    for c in df_raw.columns
])

null_summary.show(truncate=False)

+----------+----------------------+-----------------+------------------+----------------+---------------+---------+--------------------+-----------+----------+----------------------+------+-----------------------+---------+------------------+--------------+--------------+-----------------------+---------------------------+-----------------------------+----------------------+----------+-----------------+-----------+
|crash_date|traffic_control_device|weather_condition|lighting_condition|first_crash_type|trafficway_type|alignment|roadway_surface_cond|road_defect|crash_type|intersection_related_i|damage|prim_contributory_cause|num_units|most_severe_injury|injuries_total|injuries_fatal|injuries_incapacitating|injuries_non_incapacitating|injuries_reported_not_evident|injuries_no_indication|crash_hour|crash_day_of_week|crash_month|
+----------+----------------------+-----------------+------------------+----------------+---------------+---------+--------------------+-----------+----------+---

## 11 — Pemeriksaan duplikasi

In [11]:
duplicate_count = (
    total_rows - df_raw.dropDuplicates().count()
)

print("Total Baris       :", total_rows)
print("Total Duplikasi   :", duplicate_count)

Total Baris       : 209306
Total Duplikasi   : 31


## 12 — Pemeriksaan kategori khusus

In [12]:
# Mengidentifikasi kolom bertipe string
categorical_columns = [
    field.name
    for field in df_raw.schema.fields
    if field.dataType.simpleString() == "string"
]

print("Kolom Kategorikal:")
print(categorical_columns)

Kolom Kategorikal:
['crash_date', 'traffic_control_device', 'weather_condition', 'lighting_condition', 'first_crash_type', 'trafficway_type', 'alignment', 'roadway_surface_cond', 'road_defect', 'crash_type', 'intersection_related_i', 'damage', 'prim_contributory_cause', 'most_severe_injury']


In [13]:
# Menghitung kategori khusus pada setiap kolom kategorikal
special_values = [
    "UNKNOWN",
    "OTHER",
    "NOT APPLICABLE"
]

for column_name in categorical_columns:
    print(f"\nKolom: {column_name}")

    (
        df_raw
        .filter(
            lower(trim(col(column_name))).isin(
                [value.lower() for value in special_values]
            )
        )
        .groupBy(column_name)
        .count()
        .show()
    )


Kolom: crash_date
+----------+-----+
|crash_date|count|
+----------+-----+
+----------+-----+


Kolom: traffic_control_device
+----------------------+-----+
|traffic_control_device|count|
+----------------------+-----+
|               UNKNOWN| 4455|
|                 OTHER|  670|
+----------------------+-----+


Kolom: weather_condition
+-----------------+-----+
|weather_condition|count|
+-----------------+-----+
|          UNKNOWN| 6534|
|            OTHER|  627|
+-----------------+-----+


Kolom: lighting_condition
+------------------+-----+
|lighting_condition|count|
+------------------+-----+
|           UNKNOWN| 4336|
+------------------+-----+


Kolom: first_crash_type
+----------------+-----+
|first_crash_type|count|
+----------------+-----+
+----------------+-----+


Kolom: trafficway_type
+---------------+-----+
|trafficway_type|count|
+---------------+-----+
|        UNKNOWN| 1060|
|          OTHER| 4757|
+---------------+-----+


Kolom: alignment
+---------+-----+
|alignmen

## 13 — Pemeriksaan rentang nilai numerik

In [14]:
numeric_columns = [
    field.name
    for field in df_raw.schema.fields
    if field.dataType.simpleString() in [
        "int", "bigint", "double", "float", "smallint"
    ]
]

print("Kolom Numerik:")
print(numeric_columns)

df_raw.select(numeric_columns).describe().show(truncate=False)

Kolom Numerik:
['num_units', 'injuries_total', 'injuries_fatal', 'injuries_incapacitating', 'injuries_non_incapacitating', 'injuries_reported_not_evident', 'injuries_no_indication', 'crash_hour', 'crash_day_of_week', 'crash_month']
+-------+------------------+-------------------+--------------------+-----------------------+---------------------------+-----------------------------+----------------------+------------------+------------------+-----------------+
|summary|num_units         |injuries_total     |injuries_fatal      |injuries_incapacitating|injuries_non_incapacitating|injuries_reported_not_evident|injuries_no_indication|crash_hour        |crash_day_of_week |crash_month      |
+-------+------------------+-------------------+--------------------+-----------------------+---------------------------+-----------------------------+----------------------+------------------+------------------+-----------------+
|count  |209306            |209306             |209306              |209306

## 14 — Ringkasan hasil profiling

In [15]:
print("=" * 55)
print("DATA PROFILING SUMMARY")
print("=" * 55)

print(f"Dataset Name       : traffic_accidents.csv")
print(f"Total Rows         : {total_rows:,}")
print(f"Total Columns      : {total_columns}")
print(f"Duplicate Rows     : {duplicate_count:,}")
print(f"Categorical Cols   : {len(categorical_columns)}")
print(f"Numeric Cols       : {len(numeric_columns)}")

print("=" * 55)
print("DATA INGESTION & PROFILING COMPLETED")
print("=" * 55)

DATA PROFILING SUMMARY
Dataset Name       : traffic_accidents.csv
Total Rows         : 209,306
Total Columns      : 24
Duplicate Rows     : 31
Categorical Cols   : 14
Numeric Cols       : 10
DATA INGESTION & PROFILING COMPLETED
